# Support Vector Regression (SVR)

This notebook demonstrates SVR with Scikit-learn's Diabetes dataset. It covers baseline regression, Linear and Polynomial kernels, GridSearchCV, and LinearSVR. The workflow standardizes input features and compares models using R2; the final section reports errors in the original target units.

In [ ]:
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.svm import SVR, LinearSVR
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
df = datasets.load_diabetes(as_frame=True).frame

In [ ]:
df.head()
df.shape

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

In [ ]:
# Standardize the regression target using training data only.
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.to_numpy().reshape(-1, 1)).ravel()
y_test_scaled = y_scaler.transform(y_test.to_numpy().reshape(-1, 1)).ravel()

## Scale input features

SVM regressors are sensitive to feature magnitudes. The feature scaler is fit on training data only. For cross-validation, a pipeline below ensures scaling is refit inside each fold.

In [ ]:
X_scaler = StandardScaler()
X_train_scaled = X_scaler.fit_transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

In [ ]:
model = SVR()
model.fit(X_train_scaled, y_train_scaled)

In [ ]:
y_test_pred_scaled = model.predict(X_test_scaled)
y_train_pred_scaled = model.predict(X_train_scaled)

In [ ]:
print("Train R2:", r2_score(y_train_scaled, y_train_pred_scaled))
print("Test R2:", r2_score(y_test_scaled, y_test_pred_scaled))

In [ ]:
# Linear kernel
model = SVR(kernel="linear")
model.fit(X_train_scaled, y_train_scaled)
y_test_pred_scaled = model.predict(X_test_scaled)
y_train_pred_scaled = model.predict(X_train_scaled)
print("Train R2:", r2_score(y_train_scaled, y_train_pred_scaled))
print("Test R2:", r2_score(y_test_scaled, y_test_pred_scaled))

In [ ]:
# Polynomial kernel
model = SVR(kernel="poly")
model.fit(X_train_scaled, y_train_scaled)
y_test_pred_scaled = model.predict(X_test_scaled)
y_train_pred_scaled = model.predict(X_train_scaled)
print("Train R2:", r2_score(y_train_scaled, y_train_pred_scaled))
print("Test R2:", r2_score(y_test_scaled, y_test_pred_scaled))

In [ ]:
print("train r2: ", r2_score(y_train_scaled, y_train_pred_scaled))
print("test r2: ", r2_score(y_test_scaled, y_test_pred_scaled))

In [ ]:
param_grid = {
    "svr__C": [1, 2, 5, 10, 50, 100],
    "svr__kernel": ["rbf", "linear"],
    "svr__epsilon": [0.01, 0.1, 0.2, 0.3, 0.5]
}

In [ ]:
svr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR())
])

grid_search = GridSearchCV(
    svr_pipeline,
    param_grid,
    scoring="r2",
    cv=5,
    n_jobs=-1
)
grid_search.fit(X_train, y_train_scaled)

In [ ]:
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation R2:", grid_search.best_score_)

In [ ]:
best_model = grid_search.best_estimator_
y_test_pred_scaled = best_model.predict(X_test)
y_train_pred_scaled = best_model.predict(X_train)
print("Train R2:", r2_score(y_train_scaled, y_train_pred_scaled))
print("Test R2:", r2_score(y_test_scaled, y_test_pred_scaled))

In [ ]:
linear_svr = make_pipeline(
    StandardScaler(),
    LinearSVR(C=10, epsilon=0.1, max_iter=10000, random_state=42)
)
linear_svr.fit(X_train, y_train_scaled)
y_test_pred_scaled = linear_svr.predict(X_test)
y_train_pred_scaled = linear_svr.predict(X_train)
print("Train R2:", r2_score(y_train_scaled, y_train_pred_scaled))
print("Test R2:", r2_score(y_test_scaled, y_test_pred_scaled))

In [ ]:
from sklearn.pipeline import Pipeline

svr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR())
])

grid_search = GridSearchCV(
    svr_pipeline,
    param_grid,
    scoring="r2",
    cv=5,
    n_jobs=-1
)
grid_search.fit(X_train, y_train_scaled)

In [ ]:
print("best params - ", grid_search.best_params_)
print("best cross-validation R2 - ", grid_search.best_score_)

In [ ]:
best_model = grid_search.best_estimator_

y_test_pred_scaled = best_model.predict(X_test)
y_train_pred_scaled = best_model.predict(X_train)

print("train R2: ", r2_score(y_train_scaled, y_train_pred_scaled))
print("test R2: ", r2_score(y_test_scaled, y_test_pred_scaled))

In [ ]:
from sklearn.svm import LinearSVR
from sklearn.pipeline import make_pipeline

linear_svr = make_pipeline(
    StandardScaler(),
    LinearSVR(C=10, epsilon=0.1, max_iter=10000, random_state=42)
)
linear_svr.fit(X_train, y_train_scaled)

y_test_pred_scaled = linear_svr.predict(X_test)
y_train_pred_scaled = linear_svr.predict(X_train)

print("train R2: ", r2_score(y_train_scaled, y_train_pred_scaled))
print("test R2: ", r2_score(y_test_scaled, y_test_pred_scaled))

## Regression metrics in the original target units

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Predictions are inverse-transformed to the original Diabetes target scale.
y_test_pred = y_scaler.inverse_transform(y_test_pred_scaled.reshape(-1, 1)).ravel()
y_test_original = y_test.to_numpy()

print("MAE:", mean_absolute_error(y_test_original, y_test_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test_original, y_test_pred)))
print("R2:", r2_score(y_test_scaled, y_test_pred_scaled))

## Regression metrics in the original target units

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Use predictions from the selected GridSearchCV estimator.
y_test_pred = y_scaler.inverse_transform(y_test_pred_scaled.reshape(-1, 1)).ravel()
y_test_original = y_test.to_numpy()

print("MAE:", mean_absolute_error(y_test_original, y_test_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test_original, y_test_pred)))
print("R2:", r2_score(y_test_scaled, y_test_pred_scaled))